In [13]:
import trimesh

box = trimesh.creation.box(extents=[10, 15, 3])
print(box.bounds)
print(box.volume)

[[-5.  -7.5 -1.5]
 [ 5.   7.5  1.5]]
450.0


In [14]:
print(box.vertices.shape)
print(box.faces.shape)
print(box.vertices[:3])

(8, 3)
(12, 3)
[[-5.  -7.5 -1.5]
 [-5.  -7.5  1.5]
 [-5.   7.5 -1.5]]


In [15]:
import plotly.graph_objects as go

def show_mesh(mesh):
    v = mesh.vertices
    f = mesh.faces
    fig = go.Figure(data=[
        go.Mesh3d(
            x=v[:, 0], y=v[:, 1], z=v[:, 2],
            i=f[:, 0], j=f[:, 1], k=f[:, 2],
            color="lightblue",
            flatshading=True,
            opacity=1.0,
        )
    ])
    fig.update_layout(scene=dict(aspectmode="data"), margin=dict(l=0, r=0, t=0, b=0))
    fig.show()

show_mesh(box)

In [16]:
def make_box(x, y, z, width, length, height):
    box = trimesh.creation.box(extents=[width, length, height])
    box.apply_translation([x + width / 2, y + length / 2, z + height / 2])
    return box

test = make_box(0, 0, 0, 10, 15, 3)
print(test.bounds)

[[ 0.  0.  0.]
 [10. 15.  3.]]


In [17]:
def make_house(width, length, height, wall=0.3, slab=0.3):
    parts = []

    # пол
    parts.append(make_box(0, 0, 0, width, length, slab))

    # четыре стены
    parts.append(make_box(0, 0, slab, width, wall, height))                      # южная
    parts.append(make_box(0, length - wall, slab, width, wall, height))          # северная
    parts.append(make_box(0, wall, slab, wall, length - 2 * wall, height))       # западная
    parts.append(make_box(width - wall, wall, slab, wall, length - 2 * wall, height))  # восточная

    # плоская крыша
    parts.append(make_box(0, 0, slab + height, width, length, slab))

    return trimesh.util.concatenate(parts)

house = make_house(10, 15, 3)
print(house.bounds)
show_mesh(house)

[[ 0.   0.   0. ]
 [10.  15.   3.6]]


In [18]:
print(house.bounds)
print(house.bounds[1][2])

[[ 0.   0.   0. ]
 [10.  15.   3.6]]
3.5999999999999996


In [19]:
def make_wall_with_window(wall_x, wall_y, wall_z, wall_width, wall_thickness, wall_height,
                          win_x, win_width, win_sill, win_height):
    parts = []
    # левая часть стены (до окна)
    parts.append(make_box(wall_x, wall_y, wall_z, win_x, wall_thickness, wall_height))
    # правая часть стены (после окна)
    right_start = win_x + win_width
    parts.append(make_box(wall_x + right_start, wall_y, wall_z,
                          wall_width - right_start, wall_thickness, wall_height))
    # под окном
    parts.append(make_box(wall_x + win_x, wall_y, wall_z,
                          win_width, wall_thickness, win_sill))
    # над окном
    top_start = win_sill + win_height
    parts.append(make_box(wall_x + win_x, wall_y, wall_z + top_start,
                          win_width, wall_thickness, wall_height - top_start))
    return trimesh.util.concatenate(parts)

test_wall = make_wall_with_window(0, 0, 0.3, 10, 0.3, 3, 3, 4, 0.9, 1.5)
print(round(test_wall.volume, 2))
show_mesh(test_wall)

7.2


In [20]:
def make_house(width, length, height, wall=0.3, slab=0.3, south_window=None):
    parts = []

    # пол
    parts.append(make_box(0, 0, 0, width, length, slab))

    # южная стена: с окном или сплошная
    if south_window is not None:
        parts.append(make_wall_with_window(
            0, 0, slab, width, wall, height,
            south_window["x"], south_window["width"],
            south_window["sill"], south_window["height"],
        ))
    else:
        parts.append(make_box(0, 0, slab, width, wall, height))

    # остальные стены
    parts.append(make_box(0, length - wall, slab, width, wall, height))
    parts.append(make_box(0, wall, slab, wall, length - 2 * wall, height))
    parts.append(make_box(width - wall, wall, slab, wall, length - 2 * wall, height))

    # плоская крыша
    parts.append(make_box(0, 0, slab + height, width, length, slab))

    return trimesh.util.concatenate(parts)


def make_glass(window, wall=0.3, slab=0.3):
    # тонкая пластина (4 см) посередине толщины стены
    return make_box(window["x"], wall / 2 - 0.02, slab + window["sill"],
                    window["width"], 0.04, window["height"])


window = {"x": 3, "width": 4, "sill": 0.9, "height": 1.5}
house = make_house(10, 15, 3, south_window=window)
glass = make_glass(window)

print(round(house.volume, 2))
print("Площадь окна:", window["width"] * window["height"], "м²")

132.12
Площадь окна: 6.0 м²


In [21]:
def show_scene(items):
    # items: список троек (меш, цвет, прозрачность)
    traces = []
    for mesh, color, opacity in items:
        v = mesh.vertices
        f = mesh.faces
        traces.append(go.Mesh3d(
            x=v[:, 0], y=v[:, 1], z=v[:, 2],
            i=f[:, 0], j=f[:, 1], k=f[:, 2],
            color=color, flatshading=True, opacity=opacity,
        ))
    fig = go.Figure(data=traces)
    fig.update_layout(scene=dict(aspectmode="data"), margin=dict(l=0, r=0, t=0, b=0))
    fig.show()

show_scene([(house, "lightgray", 1.0), (glass, "deepskyblue", 0.5)])

In [22]:
def build_from_spec(spec):
    window = spec.get("south_window")
    house = make_house(spec["width"], spec["length"], spec["height"], south_window=window)
    glass = make_glass(window) if window else None
    return house, glass


spec = {
    "width": 8,
    "length": 12,
    "height": 2.8,
    "south_window": {"x": 1.5, "width": 5, "sill": 0.8, "height": 1.6},
}

house2, glass2 = build_from_spec(spec)

items = [(house2, "lightgray", 1.0)]
if glass2 is not None:
    items.append((glass2, "deepskyblue", 0.5))

print("Площадь пола:", spec["width"] * spec["length"], "м²")
print("Площадь окна:", spec["south_window"]["width"] * spec["south_window"]["height"], "м²")
show_scene(items)

Площадь пола: 96 м²
Площадь окна: 8.0 м²


In [23]:
spec_no_window = {"width": 6, "length": 6, "height": 3}
house3, glass3 = build_from_spec(spec_no_window)
show_scene([(house3, "lightgray", 1.0)])

In [24]:
import json

fake_ai_answer = """{
  "width": 8,
  "length": 12,
  "height": 2.8,
  "south_window": {"x": 1.5, "width": 5, "sill": 0.8, "height": 1.6}
}"""

spec_from_text = json.loads(fake_ai_answer)
print(type(spec_from_text))
print(spec_from_text["width"])

<class 'dict'>
8


In [25]:
def extract_json(text):
    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1:
        raise ValueError("В ответе нет JSON")
    return json.loads(text[start:end + 1])


messy = """Конечно! Вот описание здания:
```json
{"width": 8, "length": 12, "height": 2.8}
```
Надеюсь, это поможет!"""

print(extract_json(messy))

{'width': 8, 'length': 12, 'height': 2.8}


In [26]:
def validate_spec(spec):
    errors = []

    # 1. обязательные поля
    for key in ["width", "length", "height"]:
        if key not in spec:
            errors.append(f"Нет поля '{key}'")
    if errors:
        return errors  # дальше проверять нечего

    # 2. размеры должны быть числами и в разумных пределах
    limits = {"width": (2, 50), "length": (2, 50), "height": (2, 6)}
    for key, (low, high) in limits.items():
        value = spec[key]
        if not isinstance(value, (int, float)):
            errors.append(f"'{key}' должно быть числом, а не {value!r}")
        elif not (low <= value <= high):
            errors.append(f"'{key}' = {value}: допустимо от {low} до {high} м")
    if errors:
        return errors

    # 3. окно
    window = spec.get("south_window")
    if window is not None:
        for key in ["x", "width", "sill", "height"]:
            if key not in window:
                errors.append(f"В окне нет поля '{key}'")
        if not errors:
            if window["x"] < 0 or window["width"] <= 0 or window["sill"] < 0 or window["height"] <= 0:
                errors.append("Размеры окна должны быть положительными")
            if window["x"] + window["width"] > spec["width"]:
                errors.append("Окно шире стены")
            if window["sill"] + window["height"] > spec["height"]:
                errors.append("Окно выше стены")

    return errors

In [27]:
good = {"width": 8, "length": 12, "height": 2.8,
        "south_window": {"x": 1.5, "width": 5, "sill": 0.8, "height": 1.6}}
bad1 = {"width": -5, "length": 12, "height": 2.8}
bad2 = {"width": 8, "length": "12 м", "height": 2.8}
bad3 = {"width": 8, "length": 12, "height": 2.8,
        "south_window": {"x": 5, "width": 6, "sill": 0.8, "height": 1.6}}
bad4 = {"width": 8, "length": 12}

for name, s in [("good", good), ("bad1", bad1), ("bad2", bad2), ("bad3", bad3), ("bad4", bad4)]:
    print(name, "->", validate_spec(s))

good -> []
bad1 -> ["'width' = -5: допустимо от 2 до 50 м"]
bad2 -> ["'length' должно быть числом, а не '12 м'"]
bad3 -> ['Окно шире стены']
bad4 -> ["Нет поля 'height'"]


In [28]:
def text_to_building(ai_text):
    # 1. достаём JSON из текста
    try:
        spec = extract_json(ai_text)
    except (ValueError, json.JSONDecodeError):
        return None, None, ["ИИ вернул что-то непонятное, JSON не найден"]

    # 2. проверяем
    errors = validate_spec(spec)
    if errors:
        return None, None, errors

    # 3. строим
    house, glass = build_from_spec(spec)
    return house, glass, []

In [29]:
def run(ai_text):
    house, glass, errors = text_to_building(ai_text)
    if errors:
        print("Не получилось построить:")
        for e in errors:
            print(" -", e)
        return
    items = [(house, "lightgray", 1.0)]
    if glass is not None:
        items.append((glass, "deepskyblue", 0.5))
    show_scene(items)

In [31]:
fence = "`" * 3  # три обратные кавычки, собранные кодом

answer1 = (
    "Конечно! Вот здание:\n"
    + fence + "json\n"
    + '{"width": 8, "length": 12, "height": 2.8, '
    + '"south_window": {"x": 1.5, "width": 5, "sill": 0.8, "height": 1.6}}\n'
    + fence + "\n"
)

# 1. хороший ответ с лишним текстом вокруг
run(answer1)

# 2. плохой ответ: окно шире стены
run('{"width": 8, "length": 12, "height": 2.8, "south_window": {"x": 5, "width": 6, "sill": 0.8, "height": 1.6}}')

# 3. вообще не JSON
run("Извините, я не могу помочь с этим запросу.")

Не получилось построить:
 - Окно шире стены
Не получилось построить:
 - ИИ вернул что-то непонятное, JSON не найден
